# DLGenAI Milestone 3
Run the cells in order. Each question has a separate cell that prints the answer to submit.

In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [17]:
!pip install -q faiss-cpu sentence-transformers transformers scikit-learn

In [18]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline


In [19]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("Creating Knowledge Base...")
kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = embed_model.encode(kb, show_progress_bar=True)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge Base Ready!")


Creating Knowledge Base...
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Knowledge Base Ready!


In [20]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row150 = train.iloc[150]

prompt150 = str(row150["prompt"])

labels150 = [
    str(row150["A"]),
    str(row150["B"]),
    str(row150["C"]),
    str(row150["D"]),
    str(row150["E"])
]

correct_option = str(row150[row150["answer"]])


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

## Q1

In [21]:
result = zs(prompt150, candidate_labels=labels150)

score = result["scores"][result["labels"].index(correct_option)]

print(round(score,3))


0.384


## Q2

In [22]:
query_embedding = embed_model.encode([prompt150])

distances, indices = index.search(query_embedding, 10)

rank = None

for position, idx in enumerate(indices[0], start=1):
    if idx == 150:
        rank = position
        break

print(rank)


10


## Q3

In [23]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs = [kb[i] for i in indices[0]]

pairs = [[prompt150, doc] for doc in docs]

scores = cross_encoder.predict(pairs)

order = np.argsort(scores)[::-1]

rank = None

for position, idx in enumerate(order, start=1):
    if indices[0][idx] == 150:
        rank = position
        break

print(rank)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

1


## Q4

In [24]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row42 = train.iloc[42]
prompt42 = str(row42["prompt"])

embedding = embed_model.encode([prompt42])

_, retrieved = index.search(embedding,5)

docs = [kb[i] for i in retrieved[0]]

context = " ".join(docs)

rag = f"Context: {context} Question: {prompt42}"

tokens = tokenizer(rag)

print(len(tokens["input_ids"]))


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


## Q5

In [25]:
true_doc = kb[150]

rag = f"Context: {true_doc} Question: {prompt150}"

result = zs(rag, candidate_labels=labels150)

score = result["scores"][result["labels"].index(correct_option)]

print(round(score,3))


0.989


## Q6

In [26]:
wrong_doc = kb[999]

rag = f"Context: {wrong_doc} Question: {prompt150}"

result = zs(rag, candidate_labels=labels150)

score = result["scores"][result["labels"].index(correct_option)]

print(round(score,3))


0.529


## Q7

In [27]:
hits = 0

for i in range(100):
    row = train.iloc[i]
    prompt = str(row["prompt"])

    emb = embed_model.encode([prompt])

    _, retrieved = index.search(emb,5)

    retrieved_docs = [kb[idx] for idx in retrieved[0]]

    correct_doc = str(row[row["answer"]])

    if correct_doc in retrieved_docs:
        hits += 1

hit_rate = hits / 100 * 100

print(round(hit_rate,1))


73.0


## Q8

In [28]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

letters = ["A","B","C","D","E"]

def map3(actual, predicted):
    for rank, p in enumerate(predicted[:3], start=1):
        if p == actual:
            return 1/rank
    return 0

scores = []

for i in range(20):
    row = train.iloc[i]
    prompt = str(row["prompt"])

    emb = embed_model.encode([prompt])

    _, retrieved = index.search(emb,5)

    docs = [kb[idx] for idx in retrieved[0]]

    pairs = [[prompt, doc] for doc in docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    rag = f"Context: {best_doc} Question: {prompt}"

    options = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    result = zs(rag, candidate_labels=options)

    predicted = [letters[options.index(label)] for label in result["labels"]]

    scores.append(map3(row["answer"], predicted))

print(round(np.mean(scores),3))


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.975
